# ACT fresh-scene comparison execution backup
Completed evaluation. Outputs are in this Drive folder. Do not run all again: the runner rejects existing result directories.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
base=Path('/content/drive/MyDrive/act-pusht-runs')
print('RUNS',[p.name for p in base.iterdir()])
run=base/'act-seed0-20260917-l4'
print('COMPLETED',(run/'completed.json').read_text() if (run/'completed.json').exists() else 'NO')
print('CANDIDATES',[(p.name,p.stat().st_size) for p in (run/'evaluation/candidates').glob('*.pt')])
print('SELECTION',(run/'selection.json').read_text()[-6500:] if (run/'selection.json').exists() else 'NO')

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
from google.colab import userdata
repo=Path('/content/act-pusht-review')
if not repo.exists():
    authenv=os.environ.copy()
    authenv['GH_TOKEN']=userdata.get('GH_TOKEN')
    helper=Path('/tmp/act-review-askpass.py')
    helper.write_text('#!'+sys.executable+'\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["GH_TOKEN"])\n')
    helper.chmod(0o700)
    authenv['GIT_ASKPASS']=str(helper)
    authenv['GIT_TERMINAL_PROMPT']='0'
    subprocess.run(['git','-c','credential.helper=','clone','https://github.com/imwaterhuang/act-pusht.git',str(repo)],env=authenv,check=True)
    authenv.pop('GH_TOKEN',None)
    helper.unlink()
subprocess.run(['git','-C',str(repo),'checkout','53dddff105b664612bedcf150cbc1f1fd2542bb6'],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.','--no-deps'],check=True)
print('REVIEW_ENV_READY',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

In [ ]:
from pathlib import Path
import os,sys,subprocess
review_script=Path('/content/evaluate_three_checkpoints.py')
review_script.write_text("import os\nos.environ.setdefault('SDL_VIDEODRIVER','dummy')\nos.environ.setdefault('PYGAME_HIDE_SUPPORT_PROMPT','1')\nos.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'\nimport sys,json,hashlib,shutil,datetime,subprocess,platform,time\nfrom pathlib import Path\nimport torch\nfrom mini_wam.evaluation.act import generate_act_scene_split,load_act_checkpoint,evaluate_act_policy\nrepo=Path('/content/act-pusht-review')\nrun=Path('/content/drive/MyDrive/act-pusht-runs/act-seed0-20260917-l4')\nout=Path('/content/act-review-20260918-50')\nmirror=run.parent/'review-20260918-50'\nassert not out.exists() and not mirror.exists(), 'Refuse duplicate review output'\nassert json.loads((run/'completed.json').read_text())['step']==40000\nassert torch.cuda.is_available()\ntorch.set_num_threads(4)\ntorch.backends.cudnn.benchmark=False\ntorch.backends.cudnn.deterministic=True\ntorch.use_deterministic_algorithms(True)\nsha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()\nselection=json.loads((run/'selection.json').read_text())\nselection_sha=sha(run/'selection.json')\nrecords={r['step']:r for r in selection['candidates']}\nsteps=[30000,35000,40000]\nout.mkdir()\n(out/'weights').mkdir()\ncheckpoints={}\nfor step in steps:\n    src=run/records[step]['checkpoint']\n    target=out/'weights'/src.name\n    shutil.copy2(src,target)\n    assert sha(target)==records[step]['checkpoint_sha256']\n    checkpoints[step]=target\nexcluded=[repo/'splits/act_development_scenes.json']\nif (repo/'splits/dev_scenes.json').exists(): excluded.append(repo/'splits/dev_scenes.json')\nfor p in (run.parent/'validation-20260917-154249/interface').rglob('*scenes*.json'):\n    if isinstance(json.loads(p.read_text()),dict) and 'scenes' in json.loads(p.read_text()): excluded.append(p)\nscene_path=out/'scenes.json'\nscenes=generate_act_scene_split(scene_path,split='review',generation_seed=2026091801,count=50,excluded_paths=tuple(excluded))\nnew_seeds={s['environment_seed'] for s in scenes['scenes']}\nnew_states={tuple(sorted(s['state'].items())) for s in scenes['scenes']}\nfor p in excluded:\n    old=json.loads(p.read_text())['scenes']\n    assert new_seeds.isdisjoint(s['environment_seed'] for s in old)\n    assert new_states.isdisjoint(tuple(sorted(s['state'].items())) for s in old)\nmirror.mkdir()\nshutil.copy2(scene_path,mirror/'scenes.json')\nshutil.copy2(__file__,mirror/'evaluate_three_checkpoints.py')\nmanifest={'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'source_commit':subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip(),'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'python':platform.python_version(),'training_seed':0,'steps':steps,'scene_count':50,'generation_seed':2026091801,'scenes_hash':scenes['scenes_hash'],'excluded_scene_files':[str(p) for p in excluded],'disjoint_seeds_and_states':True,'original_best_step':selection['best_step'],'original_selection_sha256':selection_sha,'protocol':{'max_steps':300,'execute_steps':4,'success':'coverage > 0.95','inference_latent':'z=0'},'checkpoint_sha256':{str(s):sha(checkpoints[s]) for s in steps},'purpose':'User-requested paired comparison on 50 fresh scenes; original development selection is unchanged.'}\n(out/'manifest.json').write_text(json.dumps(manifest,indent=2))\nshutil.copy2(out/'manifest.json',mirror/'manifest.json')\nprint('REVIEW_SCENES_FROZEN',json.dumps(manifest),flush=True)\nrows=[]\nfor step in steps:\n    started=time.monotonic()\n    model,norm,payload=load_act_checkpoint(checkpoints[step],repo/'artifacts/act/data_audit.json',torch.device('cuda'))\n    assert payload['step']==step\n    print('REVIEW_STEP_STARTED',step,flush=True)\n    result=evaluate_act_policy(model,norm,scene_path,out/f'step_{step}',device=torch.device('cuda'),expected_split='review',max_steps=300,execute_steps=4)\n    assert len(result['episodes'])==50 and result['scenes_hash']==scenes['scenes_hash']\n    row={'step':step,'checkpoint_sha256':sha(checkpoints[step]),'seconds':time.monotonic()-started,**result['summary']}\n    rows.append(row)\n    shutil.copytree(out/f'step_{step}',mirror/f'step_{step}')\n    (out/'summary.json').write_text(json.dumps({'manifest':manifest,'results':rows,'complete':len(rows)==3},indent=2))\n    shutil.copy2(out/'summary.json',mirror/'summary.json')\n    assert sha(out/f'step_{step}/results.json')==sha(mirror/f'step_{step}/results.json')\n    print('REVIEW_STEP_COMPLETE',json.dumps(row),flush=True)\n    del model,norm,payload,result\n    torch.cuda.empty_cache()\nassert sha(run/'selection.json')==selection_sha\nprint('ACT_FRESH50_COMPARISON_COMPLETE',json.dumps(rows),flush=True)\n")
review_log=Path('/content/act-review-20260918.log')
review_env=os.environ.copy()
review_env.pop('GH_TOKEN',None)
review_env.update(OMP_NUM_THREADS='4',MKL_NUM_THREADS='4',CUBLAS_WORKSPACE_CONFIG=':4096:8')
review_handle=review_log.open('w')
review_process=subprocess.Popen([sys.executable,'-u',str(review_script)],cwd='/content/act-pusht-review',env=review_env,stdout=review_handle,stderr=subprocess.STDOUT)
print('REVIEW_STARTED',review_process.pid)

In [ ]:
import time
seen=0
while True:
    current=review_log.read_text()
    if len(current)>seen:
        print(current[seen:],end='',flush=True)
        seen=len(current)
    code=review_process.poll()
    if code is not None:
        print('REVIEW_EXIT',code,flush=True)
        assert code==0
        break
    time.sleep(15)